# Merge Rate by Contributor Tenure - Visualization

This visualization concept is tracked in:
* 8Knot Feature Request: https://github.com/oss-aspen/8Knot/issues/710
* CHAOSS Issue: https://github.com/chaoss/wg-data-science/issues/201
* CHAOSS Metric: https://chaoss.community/kb/metric-change-request-acceptance-ratio/

Description of Graphs:

First Graph:
* X-axis: Contributor Tenure Cohort (First PR, < 30 Days, 1 - 6 Months, 6 - 12 Months, 1+ Years)
* Y-axis: Number of PRs (Submitted vs Merged)

Second Graph:
* X-axis: Contributor Tenure Cohort
* Y-axis: Merge Rate in % (Merged PRs / Submitted PRs)

Third Graph:
* X-axis: Contributor Tenure Cohort
* Y-axis: Average Review Count per PR

Fourth Graph:
* X-axis: Contributor Tenure Cohort
* Y-axis: Days to Merge

The notebook does all the preprocessing down to the visualization enabling testing and analysis before implementing it on 8Knot.

### Importing Libraries

In [ ]:
import sys
import subprocess
import os
import json
from pathlib import Path
import datetime as dt
import pandas as pd
import numpy as np
import sqlalchemy as salc
import plotly.graph_objects as go
import plotly.express as px

## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install dependencies if needed

In [1]:
# Package management - Installing requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "8knot" else Path.cwd()
_req_notebook = Path(__file__).parent / "requirements.txt" if "__file__" in dir() else _repo_root / "requirements.txt"

if _req_notebook.exists():
    print("Found requirements.txt at", _req_notebook)
else:
    print("Running with existing environment packages")

Running with existing environment packages


### Loading Credentials - .env; .json...

In [2]:
# augur_creds.json lives in the same folder as this notebook (8knot/).
# If Jupyter's working directory is collectoss-analysis/ instead, fall back to 8knot/augur_creds.json.
_cred_path = Path("augur_creds.json")
if not _cred_path.is_file():
    _cred_path = Path("8knot") / "augur_creds.json"
if not _cred_path.is_file():
    raise FileNotFoundError(
        "augur_creds.json should sit next to merge_rate_by_tenure.ipynb. "
        f"Not found from cwd: {Path.cwd().resolve()}"
    )
with open(_cred_path) as config_file:
    config = json.load(config_file)

### DB connection

In [3]:
database_connection_string = "postgresql+psycopg2://{}:{}@{}:{}/{}".format(
    config["user"], config["password"], config["host"], config["port"], config["database"]
)
dbschema = "augur_data"
engine = salc.create_engine(
    database_connection_string,
    connect_args={"options": "-csearch_path={}".format(dbschema)},
)

### Queries

Get repo_ids and repo_names from augur data

In [4]:
repo_urls = ["https://github.com/chaoss/augur"]

url_query = str(repo_urls)[1:-1]
repo_query = salc.sql.text(
    f"""
    SET SCHEMA 'augur_data';
    SELECT DISTINCT r.repo_id, r.repo_name
    FROM repo r
    JOIN repo_groups rg ON r.repo_group_id = rg.repo_group_id
    WHERE r.repo_git IN ({url_query})
    """
)
with engine.connect() as conn:
    t = conn.execute(repo_query)
    results = t.all()
repo_ids = [row[0] for row in results]
repo_names = [row[1] for row in results]
print(repo_ids)
print(repo_names)

[36113]
['augur']


### Query for PRs and Contributor First Contribution Date

Getting pull requests with contributor first PR date and review counts to compute tenure at PR submission.

In [5]:
repo_statement = str(repo_ids)[1:-1]

query = salc.sql.text(f"""
    WITH first_contrib AS (
        SELECT
            pr.pr_augur_contributor_id AS cntrb_id,
            pr.repo_id,
            MIN(pr.pr_created_at) AS first_pr_at
        FROM pull_requests pr
        WHERE pr.repo_id IN ({repo_statement})
        GROUP BY pr.pr_augur_contributor_id, pr.repo_id
    ),
    pr_reviews AS (
        SELECT
            pull_request_id,
            COUNT(*) AS review_count
        FROM pull_request_reviews
        WHERE repo_id IN ({repo_statement})
        GROUP BY pull_request_id
    )
    SELECT
        r.repo_id,
        r.repo_name,
        pr.pull_request_id,
        pr.pr_src_number,
        left(pr.pr_augur_contributor_id::text, 15) AS cntrb_id,
        pr.pr_created_at AS created_at,
        pr.pr_closed_at AS closed_at,
        pr.pr_merged_at AS merged_at,
        fc.first_pr_at AS first_contribution_at,
        COALESCE(prv.review_count, 0)::int AS review_count
    FROM repo r
    JOIN pull_requests pr ON r.repo_id = pr.repo_id
    JOIN first_contrib fc ON pr.pr_augur_contributor_id = fc.cntrb_id AND pr.repo_id = fc.repo_id
    LEFT JOIN pr_reviews prv ON pr.pull_request_id = prv.pull_request_id
    WHERE r.repo_id IN ({repo_statement})
        AND pr.pr_created_at < now()
        AND (pr.pr_closed_at < now() OR pr.pr_closed_at IS NULL)
        AND (pr.pr_merged_at < now() OR pr.pr_merged_at IS NULL)
    ORDER BY pr.pr_created_at;
"""
)
df = pd.read_sql(query, con=engine)
df = df.reset_index()
df.drop("index", axis=1, inplace=True)

df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["closed_at"] = pd.to_datetime(df["closed_at"], utc=True)
df["merged_at"] = pd.to_datetime(df["merged_at"], utc=True)
df["first_contribution_at"] = pd.to_datetime(df["first_contribution_at"], utc=True)
df.head()

,repo_id,repo_name,pull_request_id,pr_src_number,cntrb_id,created_at,closed_at,merged_at,first_contribution_at,review_count
0,36113,augur,1000,1,cntrb_16,2022-01-04 00:00:00+00:00,2022-01-04 06:10:44.408340+00:00,2022-01-04 06:10:44.408340+00:00,2022-01-04 00:00:00+00:00,0
1,36113,augur,1001,2,cntrb_24,2022-01-05 00:00:00+00:00,NaT,NaT,2022-01-05 00:00:00+00:00,0
2,36113,augur,1002,3,cntrb_11,2022-01-12 00:00:00+00:00,2022-02-07 02:08:41.098102+00:00,2022-02-07 02:08:41.098102+00:00,2022-01-12 00:00:00+00:00,0
3,36113,augur,1003,4,cntrb_10,2022-01-15 00:00:00+00:00,NaT,NaT,2022-01-15 00:00:00+00:00,1
4,36113,augur,1004,5,cntrb_3,2022-01-18 00:00:00+00:00,2022-02-03 16:54:00.782049+00:00,2022-02-03 16:54:00.782049+00:00,2022-01-18 00:00:00+00:00,2


### Data Processing

Computing contributor tenure (in days) at PR creation time and grouping into cohorts:
* First Contribution (Day 0)
* < 30 Days
* 1 - 6 Months
* 6 - 12 Months
* 1+ Years

In [6]:
df["tenure_days"] = (df["created_at"] - df["first_contribution_at"]).dt.total_seconds() / 86400.0
df["is_merged"] = df["merged_at"].notna()
df["merge_duration_days"] = (df["merged_at"] - df["created_at"]).dt.total_seconds() / 86400.0

def assign_tenure_cohort(days):
    if days <= 0.0001:
        return "1. First PR (Day 0)"
    elif days < 30:
        return "2. < 30 Days"
    elif days < 180:
        return "3. 1 - 6 Months"
    elif days < 365:
        return "4. 6 - 12 Months"
    else:
        return "5. 1+ Years"

df["tenure_cohort"] = df["tenure_days"].apply(assign_tenure_cohort)
df["tenure_cohort"].value_counts().sort_index()

tenure_cohort
1. First PR (Day 0)    31
2. < 30 Days            9
3. 1 - 6 Months        48
4. 6 - 12 Months       37
5. 1+ Years            60

### Metric Aggregation

Summary table calculating total submitted PRs, merged PRs, merge rate (%), average reviews, and median time to merge.

In [7]:
tenure_summary = df.groupby("tenure_cohort").agg(
    prs_submitted=("pull_request_id", "count"),
    prs_merged=("is_merged", "sum"),
    avg_reviews=("review_count", lambda x: np.round(np.mean(x), 2)),
    median_time_to_merge=("merge_duration_days", lambda x: np.round(np.nanmedian(x), 2) if np.any(np.isfinite(x)) else 0.0)
).reset_index()

tenure_summary["merge_rate_pct"] = np.round(
    (tenure_summary["prs_merged"] / tenure_summary["prs_submitted"]) * 100, 1
)

tenure_summary = tenure_summary.sort_values(by="tenure_cohort").reset_index(drop=True)
tenure_summary

,tenure_cohort,prs_submitted,prs_merged,avg_reviews,median_time_to_merge,merge_rate_pct
0,1. First PR (Day 0),31,16,1.29,18.08,51.6
1,2. < 30 Days,9,7,1.89,7.94,77.8
2,3. 1 - 6 Months,48,34,1.69,1.91,70.8
3,4. 6 - 12 Months,37,37,1.86,1.50,100.0
4,5. 1+ Years,60,54,1.65,2.67,90.0


### Plots

## PRs Submitted vs. Merged by Tenure Cohort

Shows submitted and merged PR counts across contributor tenure groups.

In [8]:
fig_volume = go.Figure(
    [
        go.Bar(
            name="PRs Submitted",
            x=tenure_summary["tenure_cohort"],
            y=tenure_summary["prs_submitted"],
            marker_color="#2b5c8f",
            text=tenure_summary["prs_submitted"],
            textposition="auto",
            hovertemplate="<b>%{x}</b><br>Submitted: %{y}<extra></extra>",
        ),
        go.Bar(
            name="PRs Merged",
            x=tenure_summary["tenure_cohort"],
            y=tenure_summary["prs_merged"],
            marker_color="#2ca02c",
            text=tenure_summary["prs_merged"],
            textposition="auto",
            hovertemplate="<b>%{x}</b><br>Merged: %{y}<extra></extra>",
        ),
    ]
)

fig_volume.update_layout(
    title="Pull Request Volume (Submitted vs Merged) by Contributor Tenure",
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Number of PRs",
    barmode="group",
    font=dict(size=14),
    template="plotly_white",
)

fig_volume.show()

## PR Merge Rate (%) by Contributor Tenure Cohort

Compares merge acceptance rate across tenure cohorts with the repository average baseline.

In [9]:
overall_avg_rate = np.round((df["is_merged"].sum() / len(df)) * 100, 1)

fig_rate = go.Figure()

fig_rate.add_trace(
    go.Bar(
        name="Merge Rate (%)",
        x=tenure_summary["tenure_cohort"],
        y=tenure_summary["merge_rate_pct"],
        marker_color="#1f77b4",
        text=[f"{v:.1f}%" for v in tenure_summary["merge_rate_pct"]],
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Merge Rate: %{y:.1f}%<extra></extra>",
    )
)

fig_rate.add_trace(
    go.Scatter(
        name="Trend",
        x=tenure_summary["tenure_cohort"],
        y=tenure_summary["merge_rate_pct"],
        mode="lines+markers",
        line=dict(color="#ff7f0e", width=3),
        marker=dict(size=8),
        hovertemplate="<b>Trend</b>: %{y:.1f}%<extra></extra>",
    )
)

fig_rate.add_hline(
    y=overall_avg_rate,
    line_dash="dash",
    line_color="#d62728",
    annotation_text=f"Repo Avg: {overall_avg_rate}%",
    annotation_position="top left",
)

fig_rate.update_layout(
    title="PR Merge Rate (%) by Contributor Tenure Cohort",
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Merge Rate (%)",
    font=dict(size=14),
    template="plotly_white",
    yaxis=dict(range=[0, 105]),
)

fig_rate.show()

## Average Review Count per PR by Contributor Tenure

Displays the average number of reviews received per pull request across tenure cohorts.

In [10]:
fig_reviews = go.Figure(
    [
        go.Bar(
            x=tenure_summary["tenure_cohort"],
            y=tenure_summary["avg_reviews"],
            marker_color="#9467bd",
            text=tenure_summary["avg_reviews"],
            textposition="auto",
            hovertemplate="<b>%{x}</b><br>Avg Reviews: %{y}<extra></extra>",
        )
    ]
)

fig_reviews.update_layout(
    title="Average Reviews per PR by Contributor Tenure Cohort",
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Average Reviews per PR",
    font=dict(size=14),
    template="plotly_white",
)

fig_reviews.show()

## Time-to-Merge Distribution (Days)

Box plot showing the distribution of days from creation to merge across contributor tenure cohorts.

In [11]:
merged_df = df[df["is_merged"]].sort_values("tenure_cohort")

fig_box = px.box(
    merged_df,
    x="tenure_cohort",
    y="merge_duration_days",
    color="tenure_cohort",
    points="outliers",
    title="Time-to-Merge Distribution (Days) Across Contributor Tenure Cohorts",
    labels={"tenure_cohort": "Tenure Cohort", "merge_duration_days": "Days to Merge"},
    template="plotly_white",
)

fig_box.update_layout(
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Days to Merge",
    font=dict(size=14),
    showlegend=False,
)

fig_box.show()